In [ ]:
import os
import json
import random
import itertools
import math
import warnings
from tqdm.auto import tqdm
from tabulate import tabulate

import torch
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
import lightning as L

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps
from IPython.display import display, HTML

from models import get_model
from data import get_dataset
from utils.plot import *
from xai import *
from metrics import *


# plt.rcParams["font.size"] = 24
# plt.rcParams["font.family"] = "cmr10"

os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [ ]:
device = "cuda:0"
model_dir = "../configs/tweet-sentiment/base"
# model_dir = "../configs/imdb/base"
# model_dir = "../configs/politifact/base"
# model_dir = "../configs/tweet-sentiment/ablation/relu"
# model_dir = "../configs/tweet-sentiment/ablation/sigmoid"
# model_dir = "../configs/tweet-sentiment/ablation/rescale"
# model_dir = "../configs/tweet-sentiment/ablation/loss-binary"
# model_dir = "../configs/tweet-sentiment/ablation/loss-magnitude"
# model_dir = "../configs/tweet-sentiment/ablation/no-loss"
# model_dir = "../configs/tweet-sentiment/ablation/nothing"

model_config = json.load( open(os.path.join(model_dir, "config.json"), "r") )
model_metadata = json.load( open(os.path.join(model_dir, "masker/_metadata.json"), "r") )

In [ ]:
ds_train, ds_val, ds_test, num_classes = get_dataset(
    model_config["dataset_name"], 
    data_dir = "../_datasets", 
    splits = ["train", "validation", "test"],
    **model_config["dataset_args"], 
)

model = get_model(
    model_name = model_config["model_name"],
    checkpoint_path = os.path.join(model_dir, model_metadata["best_checkpoint"]),
    **model_config["model_args"]
).eval().to(device)

---
## Visualization

In [ ]:
text, label = ds_test[637]

with torch.no_grad():
    inputs = model.preprocess(text)
    logits, weights = model(inputs)
pred = torch.argmax(logits).item()
print(f"pred={pred} label={label}")

tokens = inputs["input_ids"].cpu().squeeze(0)
tokens = tokens.tolist()


inputs = (inputs["input_ids"], inputs["attention_mask"])
input_embeds = model.text_encoder.embed(inputs[0])
seq_length = inputs[1].sum().item()

explainer_our = OurAttribution(model)
attr_our = explainer_our(inputs)[0]

explainer_lig = LayerIntegratedGradientsAttribution(model, baselines=(inputs[0] * 0))
attr_lig = explainer_lig(inputs[0], target=pred, additional_forward_args=inputs[1])[0]

explainer_saliency = SaliencyAttribution(model)
attr_saliency = explainer_saliency(input_embeds, target=pred, additional_forward_args=inputs[1])[0]

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    explainer_dl = DeepLiftAttribution(model, baselines=(input_embeds * 0.0))
    attr_dl = explainer_dl(input_embeds, target=pred, additional_forward_args=inputs[1])[0]

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    baselines_dlshap = torch.cat([ model.text_encoder.embed(model.preprocess(ds_train[i][0])["input_ids"]) for i in range(20) ], dim=0)
    explainer_dlshap = DeepLiftShapAttribution(model, baselines=baselines_dlshap)
    attr_dlshap = explainer_dlshap(input_embeds, target=pred, additional_forward_args=inputs[1].unsqueeze(1))[0]

baselines_grad_shap = torch.cat([ model.text_encoder.embed(model.preprocess(ds_train[i][0])["input_ids"]) for i in range(20) ], dim=0)
explainer_grad_shap = GradientShapAttribution(model, baselines=baselines_grad_shap)
attr_grad_shap = explainer_grad_shap(input_embeds, target=pred, n_samples=100, additional_forward_args=inputs[1].unsqueeze(1))[0]

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    explainer_gbprop = GuidedBackpropAttribution(model)
    attr_gbprop = explainer_gbprop(input_embeds, target=pred, additional_forward_args=inputs[1].unsqueeze(1))[0]

for attr in [ attr_our, attr_lig, attr_saliency, attr_dl, attr_dlshap, attr_grad_shap, attr_gbprop ]:
    attr_plot = attr.mean(dim=2)[0][:seq_length]
    attr_plot = (attr_plot - attr_plot.min()) / (attr_plot.max() - attr_plot.min() + 1e-16)
    attr_plot = attr_plot.tolist()
    display_text_attribution(tokens[1:], attr_plot[1:], model.text_encoder._model.tokenizer, "<pad>")


---
## Evaluation

In [ ]:
def _accum_metrics(metrics, model_base, inputs, attribution):
    metrics["avg-drop"].accumulate(model_base, inputs, attribution)
    metrics["delete-auc"].accumulate(model_base, inputs, attribution)
    metrics["complexity"].accumulate(attribution)
    metrics["sparsity"].accumulate(attribution)

In [ ]:
L.seed_everything(42)
model_base = get_model_wrapper(model)

methods = [ "ours", "layer-ig", "saliency", "deeplift", "deeplift-shap", "gradient-shap", "guided-backprop" ]
metrics = {
    method: {
        "avg-drop": AverageDropMetric(device),
        "delete-auc": DeletionAUCMetric(device),
        "complexity": ComplexityMetric(device),
        "sparsity": SparsityMetric(device)
    } for method in methods
}


for i in tqdm(range(len(ds_test))):
    text, label = ds_test[i]

    with torch.no_grad():
        inputs = model.preprocess(text)
        logits, weights = model(inputs)
    pred = torch.argmax(logits).item()

    inputs = (inputs["input_ids"], inputs["attention_mask"])
    input_embeds = model.text_encoder.embed(inputs[0])


    explainer_our = OurAttribution(model)
    attr_our = explainer_our(inputs)[0]
    _accum_metrics(metrics["ours"], model_base, inputs, attr_our)

    explainer_lig = LayerIntegratedGradientsAttribution(model, baselines=(inputs[0] * 0))
    attr_lig = explainer_lig(inputs[0], target=pred, additional_forward_args=inputs[1])[0]
    _accum_metrics(metrics["layer-ig"], model_base, inputs, attr_lig)

    explainer_saliency = SaliencyAttribution(model)
    attr_saliency = explainer_saliency(input_embeds, target=pred, additional_forward_args=inputs[1])[0]
    _accum_metrics(metrics["saliency"], model_base, inputs, attr_saliency)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        explainer_dl = DeepLiftAttribution(model, baselines=(input_embeds * 0.0))
        attr_dl = explainer_dl(input_embeds, target=pred, additional_forward_args=inputs[1])[0]
        _accum_metrics(metrics["deeplift"], model_base, inputs, attr_dl)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        baselines_dlshap = torch.cat([ model.text_encoder.embed(model.preprocess(ds_train[i][0])["input_ids"]) for i in range(20) ], dim=0)
        explainer_dlshap = DeepLiftShapAttribution(model, baselines=baselines_dlshap)
        attr_dlshap = explainer_dlshap(input_embeds, target=pred, additional_forward_args=inputs[1].unsqueeze(1))[0]
        _accum_metrics(metrics["deeplift-shap"], model_base, inputs, attr_dlshap)

    baselines_grad_shap = torch.cat([ model.text_encoder.embed(model.preprocess(ds_train[i][0])["input_ids"]) for i in range(20) ], dim=0)
    explainer_grad_shap = GradientShapAttribution(model, baselines=baselines_grad_shap)
    attr_grad_shap = explainer_grad_shap(input_embeds, target=pred, n_samples=100, additional_forward_args=inputs[1].unsqueeze(1))[0]
    _accum_metrics(metrics["gradient-shap"], model_base, inputs, attr_grad_shap)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        explainer_gbprop = GuidedBackpropAttribution(model)
        attr_gbprop = explainer_gbprop(input_embeds, target=pred, additional_forward_args=inputs[1].unsqueeze(1))[0]
        _accum_metrics(metrics["guided-backprop"], model_base, inputs, attr_gbprop)

    torch.cuda.empty_cache()

In [ ]:
tab_out = []
for method in metrics:
    out = [ method ]
    for metric in metrics[method]:
        mean, std = metrics[method][metric].compute()
        out.append( f"{mean:.2f} +/- {std:.2f}" )
    tab_out.append( out )

print(tabulate(
    tab_out, 
    headers = ["method", *metrics["ours"]]
))

In [ ]:
json_out = {}
for method in metrics:
    json_out[method] = {}
    for metric in metrics[method]:
        mean, std = metrics[method][metric].compute()
        json_out[method][metric] = { "mean": mean, "std": std }

In [ ]:
json.dump(json_out, open(os.path.join(model_dir, "metrics.json"), "w"), indent=4)

---
## Radar plot

In [ ]:
model_dirs = [
    "../configs/tweet-sentiment/base",
    "../configs/imdb/base",
    "../configs/politifact/base"
]

with open(os.path.join(model_dirs[2], "metrics.json"), "r") as f:
    metrics = json.load(f)
    data = np.array([
        [ metrics[k]["avg-drop"]["mean"], metrics[k]["delete-auc"]["mean"], metrics[k]["complexity"]["mean"], metrics[k]["sparsity"]["mean"] ]
        for k in metrics.keys()
    ])

colors = [ "tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown", "tab:pink" ]
labels = ["Average drop", "Delete AUC", "Complexity", "Compactness (inv.)"]
methods = ["Ours", "Int. Gradients", "Saliency", "DeepLIFT", "DeepLIFT-SHAP", "Gradient-SHAP", "Guided Backprop."]
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]


axis_config = [
    { "min": data[:, 0].min(), "max": data[:, 0].max(), "ticks": np.round(np.linspace(data[:, 0].min(), data[:, 0].max(), 5)[1:-1], 1) }, 
    { "min": data[:, 1].min(), "max": data[:, 1].max(), "ticks": np.round(np.linspace(data[:, 1].min(), data[:, 1].max(), 5)[1:-1], 1) }, 
    { "min": data[:, 2].min(), "max": data[:, 2].max(), "ticks": np.round(np.linspace(data[:, 2].min(), data[:, 2].max(), 5)[1:-1]).astype(int) }, 
    { "min": data[:, 3].min(), "max": data[:, 3].max(), "ticks": np.round(np.linspace(data[:, 3].min(), data[:, 3].max(), 5)[1:-1]).astype(int) }
]

for i in range(4):
    data[:, i] = (data[:, i] - data[:, i].min()) / (data[:, i].max() - data[:, i].min())
for i in [3]:
    data[:, i] = 1 - (data[:, i] / np.max(data[:, i]))

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), subplot_kw=dict(polar=True))

for i, (angle, cfg) in enumerate(zip(angles, axis_config)):
    for j, tick in enumerate(cfg['ticks']):
        norm_tick = (tick - cfg['min']) / (cfg['max'] - cfg['min'])
        if i == 3:
            label = f"{cfg['ticks'][-j-1]:,}"
        else:
            label = f"{tick:,}"
        ax.text(angle, norm_tick, label,
                ha='center', va='center',
                fontsize=12, color='black',
                bbox= { "boxstyle": "round,pad=0.1", "fc": "none", "ec": "none" })

for values, color, method in zip(data[::-1], colors[::-1], methods[::-1]):
    values = values.tolist()
    values += values[:1]
    ax.plot(angles, values, color=color, linewidth=2, label=method, alpha=0.5)
    ax.fill(angles, values, color=color, alpha=0.25)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([])
for i, (angle, label) in enumerate(zip(angles[:-1], labels)):
    rotation = 270 if i == 0 else 90 if i == 2 else 0
    ax.text(
        angle, 1.15,
        label,
        ha='center',
        va='center',
        rotation=rotation,
    )
ax.set_yticklabels([])
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()